# AI AINAK — Notebook 1: Computer Vision Module

**Branch 1:** Object Detection, Class-Consistent Tracking, Monocular Relative Depth, and Looming Hazard Scoring.

---

## 1. Dataset Taxonomy & Asset Specification

To ensure rigorous scientific reproducibility, this notebook clearly distinguishes between training corpora, evaluation benchmarks, and demonstration inputs:

| Category | Dataset Name | Source / URL | License | Purpose in AI AINAK |
| :--- | :--- | :--- | :--- | :--- |
| **Training Dataset** | **Microsoft COCO 2017** (`train2017`) | [cocodataset.org](https://cocodataset.org/) | CC BY 4.0 | Pre-training the Ultralytics YOLOv8n object detection backbone across 80 common object classes (~118k images). *Omitted from demo download to avoid 20+ GB bandwidth.* |
| **Evaluation Dataset** | **COCO128** (`coco128`) | [Ultralytics Releases](https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip) | CC BY 4.0 | Standard 128-image validation mini-split used for offline detector benchmark verification (`mAP@0.5 = 0.6052`, `mAP@0.5:0.95 = 0.4468`, reported in `reports/vision-coco128-val.json`). *Optional download snippet provided below.* |
| **Demonstration Input** | **Paired Session `session-multimodal-01`** | Project Repository / Google Drive | Project License | Real-world walking corridor navigation recording (`video.mp4`, 2.30 MB, 100 frames @ 30.0 FPS, 3.33s duration). Paired with synchronized audio starting at $t=0.0\,	ext{s}$ per `recording.json`. |

---

## 2. Checkpoint & Demonstration Media Manifest

| Asset Name | Target Path | File Size | Origin / Acquisition Method |
| :--- | :--- | :--- | :--- |
| **YOLOv8n Weights** | `checkpoints/yolov8n.pt` | ~6.55 MB | Public official checkpoint (Ultralytics GitHub release assets). Downloaded automatically below if missing. |
| **Demonstration Video** | `datasets/recordings/session-multimodal-01/video.mp4` | ~2.30 MB | Project demonstration recording session. Located via Google Drive or starter bundle. |
| **Session Metadata** | `datasets/recordings/session-multimodal-01/recording.json` | ~837 B | Verified session clock and media pairing assertions. |


In [ ]:
# Step 1: Install Dependencies
# In Google Colab, execute this cell to install required libraries.
!pip install -q ultralytics opencv-python numpy matplotlib


In [ ]:
# Environment Detection, Google Drive Setup & Path Configuration
import os, sys, shutil, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("=" * 70)
    print("  AI AINAK: Google Colab Runtime Environment Detected")
    print("=" * 70)
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        print("  [OK] Google Drive mounted successfully at /content/drive")
    except Exception as e:
        print(f"  [NOTICE] Drive mount skipped or already mounted: {e}")

    # Standard Google Drive project folder
    DRIVE_AINAK = Path("/content/drive/MyDrive/AINAK_Colab")
    
    # Candidate project locations in priority order
    drive_candidates = [
        DRIVE_AINAK,
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), DRIVE_AINAK)
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    print("=" * 70)
    print("  AI AINAK: Local Runtime Environment Detected")
    print("=" * 70)
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"  Project Root resolved to: {PROJECT_ROOT}")

# Create standard subdirectories if not present
(PROJECT_ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "datasets" / "recordings" / "session-multimodal-01").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR = PROJECT_ROOT / "outputs"



In [ ]:
# Checkpoint & Dataset Acquisition Utilities
def download_url(url: str, dest_path: Path, desc: str = "") -> Path:
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.is_file() and dest_path.stat().st_size > 0:
        print(f"  [FOUND] {desc or dest_path.name} already present ({dest_path.stat().st_size / 1e6:.2f} MB). Skipping download.")
        return dest_path
    print(f"  [DOWNLOADING] {desc or dest_path.name} from:
    {url}")
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (AINAK-Colab-Environment)"})
        with urllib.request.urlopen(req) as resp, open(tmp_path, "wb") as f:
            shutil.copyfileobj(resp, f)
        tmp_path.replace(dest_path)
        print(f"  [DOWNLOAD COMPLETE] {dest_path.name} ({dest_path.stat().st_size / 1e6:.2f} MB)")
        return dest_path
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        raise RuntimeError(f"Failed to download {dest_path.name} from {url}: {e}")

def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return candidates[2] # Return standard checkpoints/ location as target

def find_session_file(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return candidates[0] # Return standard datasets/recordings/<session>/ location

def unpack_starter_bundle_if_available():
    bundle_candidates = [
        PROJECT_ROOT / "ainak_colab_starter_bundle.zip",
        PROJECT_ROOT / "outputs" / "ainak_colab_starter_bundle.zip",
        Path("/content/ainak_colab_starter_bundle.zip"),
    ]
    for b in bundle_candidates:
        if b.is_file() and b.stat().st_size > 0:
            print(f"  [UNPACKING] Found starter bundle at {b}. Extracting to {PROJECT_ROOT}...")
            import zipfile
            with zipfile.ZipFile(str(b), "r") as zf:
                zf.extractall(str(PROJECT_ROOT))
            print("  [UNPACK COMPLETE] Starter checkpoints and demonstration media extracted.")
            return True
    return False

def run_preflight_audit(required_items):
    print("=" * 80)
    print("  AI AINAK PRE-FLIGHT ASSET & DATASET INTEGRITY AUDIT")
    print("=" * 80)
    all_ok = True
    for item in required_items:
        path = Path(item["path"])
        min_size = item.get("min_size", 1)
        role = item.get("role", "Asset")
        desc = item.get("desc", path.name)
        
        if path.is_file() and path.stat().st_size >= min_size:
            size_mb = path.stat().st_size / 1e6
            status = "PASSED"
            print(f"  [OK] {status:<6} | {path.name:<28} | {size_mb:6.2f} MB | {role:<16} | {desc}")
        else:
            all_ok = False
            status = "FAILED"
            print(f"  [!!] {status:<6} | {path.name:<28} | MISSING   | {role:<16} | {desc}")
            print(f"       -> Expected location: {path}")
            if "download_url" in item:
                print(f"       -> Download URL: {item['download_url']}")
            elif "action" in item:
                print(f"       -> Action required: {item['action']}")
    print("=" * 80)
    if not all_ok:
        raise FileNotFoundError(
            "Pre-flight audit failed: One or more required datasets or model checkpoints are missing. "
            "Please review the missing items above and obtain them before proceeding."
        )
    print("  All required assets verified. Proceeding to inference...
")



## Step 2: Automated Dataset & Checkpoint Acquisition
Here we acquire the public YOLOv8n weights automatically and resolve the paired demonstration session.


In [ ]:
# Check for starter bundle first
unpack_starter_bundle_if_available()

# 1. Acquire YOLOv8n Checkpoint (Official Public Ultralytics Release)
YOLO_URL = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt"
yolo_path = find_checkpoint("yolov8n.pt", "vision")
if not yolo_path.is_file():
    yolo_target = PROJECT_ROOT / "checkpoints" / "yolov8n.pt"
    yolo_path = download_url(YOLO_URL, yolo_target, "YOLOv8n Checkpoint")

# 2. Resolve Demonstration Media
video_path = find_session_file("session-multimodal-01", "video.mp4")
metadata_path = find_session_file("session-multimodal-01", "recording.json")

print(f"YOLOv8n Checkpoint Path: {yolo_path}")
print(f"Video Session Path: {video_path}")
print(f"Metadata Path: {metadata_path}")


### Optional: Evaluation Dataset Acquisition (COCO128)
If you wish to benchmark the YOLOv8n detector against the COCO128 validation split (as documented in `reports/vision-coco128-val.json`), set `DOWNLOAD_COCO128_EVAL = True`. Otherwise, only the demonstration media is used.


In [ ]:
DOWNLOAD_COCO128_EVAL = False

if DOWNLOAD_COCO128_EVAL:
    coco128_zip = PROJECT_ROOT / "datasets" / "coco128.zip"
    coco128_dir = PROJECT_ROOT / "datasets" / "coco128"
    COCO128_URL = "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip"
    download_url(COCO128_URL, coco128_zip, "COCO128 Benchmark Dataset")
    if not coco128_dir.is_dir():
        import zipfile
        with zipfile.ZipFile(str(coco128_zip), "r") as zf:
            zf.extractall(str(PROJECT_ROOT / "datasets"))
        print(f"[OK] Extracted COCO128 dataset to: {coco128_dir}")
else:
    print("[INFO] Skipping COCO128 evaluation dataset download (demonstration mode active).")


## Step 3: Pre-Flight Asset Integrity & Clock Validation
We verify that all required files exist, have non-zero size, and match expected video clock properties before starting inference.


In [ ]:
import json
import cv2

# Execute pre-flight integrity audit
required_manifest = [
    {
        "path": yolo_path,
        "min_size": 6_000_000,
        "role": "Model Checkpoint",
        "desc": "Trained YOLOv8n object detector",
        "download_url": YOLO_URL
    },
    {
        "path": video_path,
        "min_size": 2_000_000,
        "role": "Demonstration Input",
        "desc": "Paired session video (session-multimodal-01)",
        "action": "Upload session-multimodal-01/video.mp4 to Google Drive or project folder"
    },
    {
        "path": metadata_path,
        "min_size": 100,
        "role": "Clock Metadata",
        "desc": "Session clock & pairing assertion",
        "action": "Ensure recording.json is present in session-multimodal-01"
    }
]
run_preflight_audit(required_manifest)

# Validate Video Clock & Properties
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_sec = total_frames / fps if fps > 0 else 0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Video Stream Properties: {width}x{height} @ {fps:.1f} FPS | Total Frames: {total_frames} | Duration: {duration_sec:.2f}s")

if metadata_path.is_file():
    with open(metadata_path, "r") as f:
        meta = json.load(f)
    print(f"Verified Pairing Assertion: {meta.get('pairing_assertion', 'N/A')}")
    print(f"Clock Spec: {meta.get('clock', 'N/A')} (Video Offset: {meta.get('video_offset_seconds', 0.0)}s)")


## Step 4: Vision Inference Implementation
Here we define the detector, depth proxy, and 128-D feature vector generator matching the AI AINAK production pipeline.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from ultralytics import YOLO

# Load model
yolo_model = YOLO(str(yolo_path))

# Hazard and Depth Scoring Logic
CRITICAL_CLASSES = {"person", "car", "truck", "bus", "motorcycle", "bicycle", "traffic light", "stop sign"}

def estimate_relative_depth(bbox, frame_shape):
    # Monocular perspective depth proxy: ground plane horizon contact + area scaling
    h, w = frame_shape[:2]
    x1, y1, x2, y2 = bbox
    base_y = y2 / h
    area = max(0.0, (x2 - x1) * (y2 - y1)) / (w * h)
    proximity = 0.65 * base_y + 0.35 * min(1.0, area * 4.0)
    return float(np.clip(proximity, 0.0, 1.0))

def compute_hazard_score(class_name, confidence, depth, delta_depth):
    base_risk = 0.8 if class_name in {"car", "truck", "bus", "motorcycle"} else 0.5 if class_name == "person" else 0.3
    loom_rate = max(0.0, delta_depth)
    hazard = (0.4 * base_risk) + (0.35 * depth) + (0.25 * min(1.0, loom_rate * 2.0))
    return float(np.clip(hazard * confidence, 0.0, 1.0))

def extract_vision_feature_128d(detections, frame_shape):
    # 128-D feature vector:
    # [0:80]   - COCO class histogram normalized by sum
    # [80:84]  - normalized [center_x, center_y, area, confidence] of primary object
    # [84:128] - zero-padded / reserved for multi-scale context
    h, w = frame_shape[:2]
    vec = np.zeros(128, dtype=np.float32)
    if not detections:
        return vec

    for d in detections:
        cls_id = int(d["class_id"]) % 80
        vec[cls_id] += float(d["confidence"])
    
    # Normalize histogram
    sum_hist = vec[:80].sum()
    if sum_hist > 0:
        vec[:80] /= sum_hist

    # Primary detection geometry
    top_d = max(detections, key=lambda x: x["confidence"])
    x1, y1, x2, y2 = top_d["bbox"]
    cx = (x1 + x2) / (2.0 * w)
    cy = (y1 + y2) / (2.0 * h)
    area = ((x2 - x1) * (y2 - y1)) / (w * h)
    vec[80:84] = [cx, cy, area, top_d["confidence"]]
    return vec


## Step 5: Process Video Frames & Track Detections
We iterate over the video, track detections with IoU, compute hazard levels, and build the 128-D features.


In [ ]:
cap = cv2.VideoCapture(str(video_path))

frame_idx = 0
timestamps = []
features = []
confidences = []
latencies_ms = []
detections_history = []
sample_annotated_frames = []

prev_tracks = [] # (bbox, class_name, track_id, depth, time)
next_track_id = 1

print("Starting Vision inference across video stream...")
t0 = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    ts = frame_idx / fps
    frame_idx += 1

    t_start = time.perf_counter()
    results = yolo_model.predict(frame, conf=0.35, iou=0.45, verbose=False, device="cpu")[0]
    t_latency = (time.perf_counter() - t_start) * 1000

    frame_dets = []
    h, w = frame.shape[:2]
    
    for i in range(len(results.boxes)):
        xyxy = results.boxes.xyxy[i].cpu().numpy().tolist()
        cls_id = int(results.boxes.cls[i].item())
        cls_name = results.names[cls_id]
        conf = float(results.boxes.conf[i].item())
        depth = estimate_relative_depth(xyxy, frame.shape)
        
        # Simple IoU track match
        best_iou = 0.3
        matched_track = None
        for prev_box, prev_name, tid, prev_depth, prev_t in prev_tracks:
            if prev_name == cls_name:
                xA = max(xyxy[0], prev_box[0])
                yA = max(xyxy[1], prev_box[1])
                xB = min(xyxy[2], prev_box[2])
                yB = min(xyxy[3], prev_box[3])
                inter = max(0, xB - xA) * max(0, yB - yA)
                area1 = (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1])
                area2 = (prev_box[2] - prev_box[0]) * (prev_box[3] - prev_box[1])
                iou = inter / max(1e-6, area1 + area2 - inter)
                if iou > best_iou:
                    best_iou = iou
                    matched_track = (tid, prev_depth, prev_t)

        if matched_track:
            tid, p_depth, p_t = matched_track
            dt = max(1e-3, ts - p_t)
            delta_d = (depth - p_depth) / dt
        else:
            tid = next_track_id
            next_track_id += 1
            delta_d = 0.0

        hazard = compute_hazard_score(cls_name, conf, depth, delta_d)

        frame_dets.append({
            "class_id": cls_id,
            "class_name": cls_name,
            "confidence": conf,
            "bbox": xyxy,
            "track_id": tid,
            "relative_depth": depth,
            "hazard_score": hazard
        })

    prev_tracks = [(d["bbox"], d["class_name"], d["track_id"], d["relative_depth"], ts) for d in frame_dets]
    feat_128 = extract_vision_feature_128d(frame_dets, frame.shape)
    
    timestamps.append(ts)
    features.append(feat_128)
    confidences.append(max((d["confidence"] for d in frame_dets), default=0.0))
    latencies_ms.append(t_latency)
    detections_history.append(frame_dets)

    # Save visual samples
    if frame_idx % 25 == 0 or frame_idx == 1:
        annotated = results.plot()
        sample_annotated_frames.append((ts, annotated, frame_dets))

cap.release()
print(f"Completed vision processing: {len(timestamps)} frames processed in {time.time() - t0:.2f}s.")
print(f"Mean Inference Latency: {np.mean(latencies_ms):.2f} ms | p95 Latency: {np.percentile(latencies_ms, 95):.2f} ms")


## Step 6: Visualization of Detections & Hazard Heatmap


In [ ]:
fig, axes = plt.subplots(1, min(3, len(sample_annotated_frames)), figsize=(16, 5))
if not isinstance(axes, np.ndarray):
    axes = [axes]

for i, (ts, frame_rgb, dets) in enumerate(sample_annotated_frames[:3]):
    img = cv2.cvtColor(frame_rgb, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(f"Time: {ts:.2f}s | Detections: {len(dets)}", fontsize=11, fontweight="bold")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## Step 7: Export Vision Outputs to `vision_outputs.npz`
We save the features, timestamps, confidences, and latencies for multimodal fusion.


In [ ]:
vision_out_path = OUTPUTS_DIR / "vision_outputs.npz"

np.savez_compressed(
    str(vision_out_path),
    timestamps=np.array(timestamps, dtype=np.float64),
    features=np.array(features, dtype=np.float32),
    confidences=np.array(confidences, dtype=np.float32),
    latencies_ms=np.array(latencies_ms, dtype=np.float32)
)

print(f"Successfully exported vision outputs to: {vision_out_path}")
print(f"File size: {vision_out_path.stat().st_size / 1e3:.1f} KB")
print(f"Feature matrix shape: {np.array(features).shape} (Matches 128-D Transformer contract)")
